<a href="https://colab.research.google.com/github/sanjaymaharja/Machine-Learning-Programming/blob/main/ASSIGNMENT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# After doing the complete Assignment making basic import in one line
#and all the visualization in last line.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*60)
print("EMPLOYEE PRODUCTIVITY PREDICTION & BEHAVIORAL SEGMENTATION")
print("="*60)

EMPLOYEE PRODUCTIVITY PREDICTION & BEHAVIORAL SEGMENTATION


In [9]:
# TODO 1: Loading dataset

print("\n" + "="*60)
print("PART 1: DATA PREPARATION")
print("="*60)

# Loading the dataset
df = pd.read_csv("assignment1_dataset1.csv")
print(f"\nDataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())

print(f"\nDataset Info:")
print(df.info())

print(f"\nBasic Statistics:")
print(df.describe())

# Checking for missing values
print(f"\nMissing Values:")
print(df.isnull().sum())

# Separating features and target variables
# Features: All columns except EmployeeID and target variables
X = df.drop(['EmployeeID', 'ProductivityScore'], axis=1)

# Target 1: ProductivityScore (Regression)
y_regression = df['ProductivityScore']

# Target 2: PerformanceCategory (Classification) - will create after encoding

# Handling categorical variables
X_encoded = pd.get_dummies(X, columns=['Gender', 'EducationLevel', 'RemoteWork'], drop_first=True)

print(f"\nFeatures after encoding: {X_encoded.shape[1]} features")
print(f"Encoded features: {list(X_encoded.columns)}")

# Performing train-test split (80/20)
X_train, X_test, y_train_reg, y_test_reg = train_test_split(
    X_encoded, y_regression, test_size=0.2, random_state=42
)

print(f"\nTrain-Test Split Results:")
print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")


PART 1: DATA PREPARATION

Dataset loaded successfully!
Shape: (20, 10)

First 5 rows:
  EmployeeID  Gender  Age  WorkHours  SleepHours  AttendanceRate  \
0       E001    Male   29          8           7              92   
1       E002  Female   34          9           6              88   
2       E003    Male   41         10           5              95   
3       E004  Female   26          7           8              85   
4       E005    Male   38          9           6              90   

   PreviousPerformance EducationLevel RemoteWork  ProductivityScore  
0                   78       Bachelor        Yes                 82  
1                   85         Master         No                 87  
2                   90            PhD         No                 92  
3                   72       Bachelor        Yes                 79  
4                   88         Master        Yes                 86  

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19

In [10]:
# TODO 2: Building and evaluating regression model

print("\n" + "="*60)
print("PART 2: SUPERVISED LEARNING - REGRESSION")
print("="*60)

# 1. Importing and training LinearRegression model
reg_model = LinearRegression()

# Training the model
reg_model.fit(X_train, y_train_reg)
print("\n✓ Linear Regression model trained successfully!")

# 2. Making predictions
y_pred_reg = reg_model.predict(X_test)

# 3. Evaluating using regression metrics
mae = mean_absolute_error(y_test_reg, y_pred_reg)
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

print("\n" + "-"*40)
print("REGRESSION MODEL EVALUATION METRICS")
print("-"*40)
print(f"Mean Absolute Error (MAE):      {mae:.4f}")
print(f"Mean Squared Error (MSE):       {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R² Score:                       {r2:.4f}")

# Featuring importance (coefficients)
feature_importance = pd.DataFrame({
    'feature': X_encoded.columns,
    'coefficient': reg_model.coef_
}).sort_values('coefficient', ascending=False)

print("\nTop 5 Most Important Features (Positive Impact):")
print(feature_importance.head(5).to_string(index=False))

print("\nTop 5 Least Important Features (Negative Impact):")
print(feature_importance.tail(5).to_string(index=False))


PART 2: SUPERVISED LEARNING - REGRESSION

✓ Linear Regression model trained successfully!

----------------------------------------
REGRESSION MODEL EVALUATION METRICS
----------------------------------------
Mean Absolute Error (MAE):      1.4360
Mean Squared Error (MSE):       2.6515
Root Mean Squared Error (RMSE): 1.6283
R² Score:                       0.9092

Top 5 Most Important Features (Positive Impact):
                feature  coefficient
     EducationLevel_PhD     5.600230
  EducationLevel_Master     3.626856
EducationLevel_Bachelor     1.608884
                    Age     0.365210
              WorkHours     0.333848

Top 5 Least Important Features (Negative Impact):
            feature  coefficient
     AttendanceRate     0.265052
PreviousPerformance     0.110254
         SleepHours     0.058753
     RemoteWork_Yes    -0.028042
        Gender_Male    -2.078105


In [11]:
# TODO 3: Building and evaluating classification model

print("\n" + "="*60)
print("PART 3: SUPERVISED LEARNING - CLASSIFICATION")
print("="*60)

# 1. Creating PerformanceCategory
def categorize_performance(score):
    if score > 85:
        return 'High Performer'
    elif score >= 70:
        return 'Average'
    else:
        return 'Low Performer'

df['PerformanceCategory'] = df['ProductivityScore'].apply(categorize_performance)

print("\nPerformance Category Distribution:")
print(df['PerformanceCategory'].value_counts())
print("\nPercentage Distribution:")
print(df['PerformanceCategory'].value_counts(normalize=True) * 100)

# Preparing data for classification
y_classification = df['PerformanceCategory']

# Spliting data for classification (using same features)
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_encoded, y_classification, test_size=0.2, random_state=42, stratify=y_classification
)

# 2. Training a classifier (Decision Tree)
clf_model = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_model.fit(X_train_clf, y_train_clf)
print("\n✓ Decision Tree Classifier trained successfully!")

# Making predictions
y_pred_clf = clf_model.predict(X_test_clf)

# 3. Evaluating using classification metrics
accuracy = accuracy_score(y_test_clf, y_pred_clf)
conf_matrix = confusion_matrix(y_test_clf, y_pred_clf)
class_report = classification_report(y_test_clf, y_pred_clf)

print("\n" + "-"*40)
print("CLASSIFICATION MODEL EVALUATION METRICS")
print("-"*40)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\nConfusion Matrix:")
print(conf_matrix)

print("\nClassification Report:")
print(class_report)

# Featuring importance from Decision Tree
feature_importance_clf = pd.DataFrame({
    'feature': X_encoded.columns,
    'importance': clf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 5 Most Important Features for Classification:")
print(feature_importance_clf.head(5).to_string(index=False))


PART 3: SUPERVISED LEARNING - CLASSIFICATION

Performance Category Distribution:
PerformanceCategory
Average           10
High Performer    10
Name: count, dtype: int64

Percentage Distribution:
PerformanceCategory
Average           50.0
High Performer    50.0
Name: proportion, dtype: float64

✓ Decision Tree Classifier trained successfully!

----------------------------------------
CLASSIFICATION MODEL EVALUATION METRICS
----------------------------------------
Accuracy: 1.0000 (100.00%)

Confusion Matrix:
[[2 0]
 [0 2]]

Classification Report:
                precision    recall  f1-score   support

       Average       1.00      1.00      1.00         2
High Performer       1.00      1.00      1.00         2

      accuracy                           1.00         4
     macro avg       1.00      1.00      1.00         4
  weighted avg       1.00      1.00      1.00         4


Top 5 Most Important Features for Classification:
            feature  importance
PreviousPerformance    0.